# SR-EV Packing Domain Analysis

Identifies packing domains in each SR-EV `config-N.txt` configuration using the method from Cangnano et al. 2024 (eLife, "Local Volume Concentration, Packing Domains and Scaling Properties of Chromatin") and Li et al. 2022 (Scientific Reports, ChromSTEM), as implemented in `hmt_v3.srev.domains`: render a Gaussian-bead density image of a 100 nm slab, find domain centers as local maxima after Gaussian smoothing + CLAHE contrast enhancement (masked to the occupied region -- see below), and size each domain from its own radial density profile.

This notebook reproduces the domain **count** and **radius distribution** across every available `(alpha, phi)` SR-EV parameter combination, and plots the radius distributions as violins grouped by `alpha` and `phi`, matching Figure 7A of Cangnano et al. 2024.

**Calibration status** (see `hmt_v3/srev/domains.py` module docstring for full detail): a bug was found and fixed (2026-09) where CLAHE's local contrast stretching manufactured spurious "domain centers" in near-zero-density void regions of the simulated density image -- real ChromSTEM tomograms never have such regions, but this simulated data does. `find_domain_centers` now masks CLAHE to an Otsu-thresholded occupied region first. Post-fix, on `hmt_v1/Simulation/config-1.txt`: 17.6 domains/µm² (Li et al. 2022 report 17.5/µm²) with every center on real density, and radius has median 85 nm / mean 107 nm (paper: A549 median Rf 74 nm, mean 80.6 nm). Both domain **count** and **radius** can now be treated as reasonable measurements rather than the count-only-trustworthy/radius-upper-bound caveat from before the fix -- **any cached `sr_ev_domain_counts.csv`/`sr_ev_domain_radii.csv` computed before this fix must be deleted (or `FORCE_RECOMPUTE = True` set below) to pick it up**, since the run-code cell's cache is keyed only by `config_name`, not by algorithm version.

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hmt_v3 import srev

## Discover config files

Edit `BASE_DIR` below to point wherever the raw `SRRW-...-config-N.txt` folders are placed in the repo (default assumes they land alongside the existing `hmt_v1/Simulation/config-1.txt`). Every `config-*.txt` found under an `SRRW-*` folder is included; the folder name is parsed for its `alpha`, `phi`, `L`, `N`, `r`, `ru` parameters via `sr_ev_io.parse_srev_folder_name`.

`SAMPLE_PER_CONDITION` caps how many config files are processed per `(alpha, phi)` combination -- set to `None` to process every file found (can take a while; see the timing note below), or a small integer (e.g. `5`) for a quick test run before committing to the full batch.

In [ ]:
BASE_DIR = Path("sr-ev_raw")  # <-- point this at wherever the SRRW-* folders live
SAMPLE_PER_CONDITION = None  # e.g. 5 for a quick test run; None = use every file found

config_paths = sorted(BASE_DIR.glob("SRRW-*/config-*.txt")) + sorted(BASE_DIR.glob("SRRW-*/config-*.txt.gz"))
print(f"found {len(config_paths)} config-N.txt(.gz) files under {BASE_DIR}/SRRW-*/")

manifest_rows = []
for path in config_paths:
    # resolve_srev_run_params prefers the folder's copy-parameters.txt (a direct record of the
    # generator's own inputs) over parsing the folder name, and cross-checks the two if both
    # exist -- see hmt_v3.srev.io for why that's more trustworthy than the folder name alone.
    try:
        params = srev.io.resolve_srev_run_params(path)
    except ValueError as exc:
        warnings.warn(f"skipping {path}: {exc}")
        continue
    if not {"alpha", "length_unit_to_nm"} <= params.keys():
        warnings.warn(f"skipping {path}: could not resolve alpha/ru from '{path.parent.name}'")
        continue
    manifest_rows.append({
        "path": path,
        "config_name": f"{path.parent.name}/{path.name}",
        "alpha": params["alpha"],
        "phi": params.get("phi"),  # only present if resolved via folder name (copy-parameters.txt doesn't carry phi -- it's derived, not a generator input)
        "L": params.get("L"),
        "N": params.get("N"),
        "length_unit_to_nm": params["length_unit_to_nm"],
    })

manifest_df = pd.DataFrame(manifest_rows)

# phi isn't in copy-parameters.txt (it's derived: phi = N*(r/Rc)^3, not a generator input), so
# recompute it from N whenever it's missing, using the fixed r=4.9nm, Rc=650nm from Cangnano et al. 2024
if len(manifest_df):
    manifest_df["phi"] = pd.to_numeric(manifest_df["phi"], errors="coerce")
    r_deg_nm, Rc_nm = 4.9, 650.0
    missing_phi = manifest_df["phi"].isna()
    manifest_df.loc[missing_phi, "phi"] = manifest_df.loc[missing_phi, "N"] * (r_deg_nm / Rc_nm) ** 3
    manifest_df["phi"] = manifest_df["phi"].round(2)

if SAMPLE_PER_CONDITION is not None and len(manifest_df):
    manifest_df = manifest_df.groupby(["alpha", "phi"], group_keys=False).head(SAMPLE_PER_CONDITION)

n_conditions = manifest_df[["alpha", "phi"]].drop_duplicates().shape[0] if len(manifest_df) else 0
print(f"processing {len(manifest_df)} configs across {n_conditions} (alpha, phi) combinations")
manifest_df.groupby(["alpha", "phi"]).size().rename("n_files").reset_index() if len(manifest_df) else manifest_df

## Run domain identification on each config

This is the slow step -- `identify_packing_domains` takes roughly 2-5 seconds per config at the default settings, so a batch of hundreds of files can take a while. Results are cached to `sr_ev_domain_counts.csv` / `sr_ev_domain_radii.csv` next to this notebook, keyed by `config_name`: rerunning only computes configs that aren't already in the cache (e.g. after widening `BASE_DIR`/`SAMPLE_PER_CONDITION` above), it never silently ignores new files. Delete the CSVs (or set `FORCE_RECOMPUTE = True`) to force a full rerun from scratch. A file that fails (e.g. an empty slab) is skipped with a warning and retried on the next run rather than stopping the whole batch.

In [ ]:
FORCE_RECOMPUTE = False

counts_path = Path("sr_ev_domain_counts.csv")
radii_path = Path("sr_ev_domain_radii.csv")

count_cols = ["config_name", "alpha", "phi", "n_domains"]
radius_cols = ["config_name", "alpha", "phi", "domain_id", "radius_nm"]
# numeric dtypes to force on every counts/radii frame below (cached, new, and their concat) --
# an empty `pd.DataFrame(columns=...)` defaults every column to object dtype, and concatenating
# that with a properly-typed float64 frame can silently upcast the merged column to object too.
# An object-dtype radius_nm/n_domains array crashes scipy's gaussian_kde (via np.cov -> np.average)
# deep inside matplotlib's violinplot with a cryptic "'float' object has no attribute 'shape'" --
# so every frame built here is cast back to these dtypes right after construction, never left to
# whatever concat happens to infer.
count_dtypes = {"alpha": "float64", "phi": "float64", "n_domains": "float64"}
radius_dtypes = {"alpha": "float64", "phi": "float64", "domain_id": "float64", "radius_nm": "float64"}

if not FORCE_RECOMPUTE and counts_path.exists() and radii_path.exists():
    cached_counts_df = pd.read_csv(counts_path).astype(count_dtypes)
    cached_radii_df = pd.read_csv(radii_path).astype(radius_dtypes)
else:
    cached_counts_df = pd.DataFrame(columns=count_cols).astype(count_dtypes)
    cached_radii_df = pd.DataFrame(columns=radius_cols).astype(radius_dtypes)

# The cache is keyed by config_name, not by "did the whole batch finish" -- a config counts
# as done once it has a row in cached_counts_df (added even when n_domains==0), so only the
# configs in the *current* manifest that are missing from the cache get (re)computed. This is
# what lets BASE_DIR/SAMPLE_PER_CONDITION grow between runs without recomputing everything.
already_done = set(cached_counts_df["config_name"])
todo_df = manifest_df[~manifest_df["config_name"].isin(already_done)].reset_index(drop=True)
print(f"{len(already_done)} configs already cached, {len(todo_df)} new configs to process this run")

count_rows = []
radius_rows = []
t_start = time.time()

for i, row in todo_df.iterrows():
    t0 = time.time()
    try:
        monomers_df, _ = srev.io.parse_config_txt(row["path"], length_unit_to_nm=row["length_unit_to_nm"])
        domains = srev.domains.identify_packing_domains(monomers_df)
    except Exception as exc:
        warnings.warn(f"skipping {row['config_name']}: {exc}")
        continue

    count_rows.append({
        "config_name": row["config_name"],
        "alpha": row["alpha"],
        "phi": row["phi"],
        "n_domains": len(domains),
    })
    for _, d in domains.iterrows():
        radius_rows.append({
            "config_name": row["config_name"],
            "alpha": row["alpha"],
            "phi": row["phi"],
            "domain_id": d["domain_id"],
            "radius_nm": d["radius_nm"],
        })

    if (i + 1) % 5 == 0 or (i + 1) == len(todo_df):
        elapsed = time.time() - t_start
        print(f"  [{i+1}/{len(todo_df)}] {row['config_name']}: "
              f"{len(domains)} domains ({time.time()-t0:.1f}s this file, {elapsed:.0f}s total)")

new_counts_df = pd.DataFrame(count_rows, columns=count_cols).astype(count_dtypes)
new_radii_df = pd.DataFrame(radius_rows, columns=radius_cols).astype(radius_dtypes)

# Full, ever-growing cache written to disk -- may contain configs outside today's manifest
# (e.g. left over from a run with a different BASE_DIR/SAMPLE_PER_CONDITION).
full_counts_df = pd.concat([cached_counts_df, new_counts_df], ignore_index=True).astype(count_dtypes)
full_radii_df = pd.concat([cached_radii_df, new_radii_df], ignore_index=True).astype(radius_dtypes)
if len(todo_df):
    full_counts_df.to_csv(counts_path, index=False)
    full_radii_df.to_csv(radii_path, index=False)

# What the rest of the notebook (summary table, violin plots) actually uses -- scoped down to
# exactly the configs the current manifest_df asked for, whether they came from cache or were
# just computed.
current_names = set(manifest_df["config_name"])
domain_counts_df = full_counts_df[full_counts_df["config_name"].isin(current_names)].reset_index(drop=True)
domain_radii_df = full_radii_df[full_radii_df["config_name"].isin(current_names)].reset_index(drop=True)

print(f"in scope for this run: {len(domain_counts_df)} configs ({len(new_counts_df)} newly computed), "
      f"{len(domain_radii_df)} domains "
      f"(full on-disk cache: {len(full_counts_df)} configs, {len(full_radii_df)} domains)")

## Domain count summary

"The number of domains in each config" -- per-`(alpha, phi)` mean and spread of `n_domains` (domains found in that config's single 100 nm slab).

In [ ]:
if len(domain_counts_df):
    summary = (
        domain_counts_df.groupby(["alpha", "phi"])["n_domains"]
        .agg(["count", "mean", "std"])
        .rename(columns={"count": "n_configs", "mean": "mean_n_domains", "std": "std_n_domains"})
        .reset_index()
    )
    display(summary)
else:
    print("no results yet -- add config files under BASE_DIR and rerun the cells above")

## Violin plots

Grouped by `alpha` (color / block) then `phi` (x position within each block) -- matching the layout of Cangnano et al. 2024 Figure 7A.

In [ ]:
def plot_grouped_violin(df, value_col, ylabel, title, save_path=None, ax=None):
    """Violin plot of `value_col`, grouped by alpha (color/block) then phi (x position within block)."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 5))
    else:
        fig = ax.figure

    alphas = sorted(df["alpha"].unique())
    phis = sorted(df["phi"].unique())
    cmap = plt.get_cmap("tab10")
    alpha_colors = {a: cmap(i) for i, a in enumerate(alphas)}

    positions, data, colors, tick_labels, group_centers = [], [], [], [], {}
    pos = 1
    for a in alphas:
        block_start = pos
        for p in phis:
            # Coerce to float64 defensively: an object-dtype array here (e.g. from an upstream
            # dtype mismatch during caching) makes scipy's gaussian_kde crash deep inside
            # violinplot with a cryptic "'float' object has no attribute 'shape'" AttributeError.
            vals = pd.to_numeric(
                df.loc[(df["alpha"] == a) & (df["phi"] == p), value_col], errors="coerce"
            ).dropna().to_numpy(dtype="float64")
            if len(vals) == 0:
                continue
            data.append(vals)
            positions.append(pos)
            colors.append(alpha_colors[a])
            tick_labels.append(f"{p:g}")
            pos += 1
        if pos > block_start:
            group_centers[a] = (block_start + pos - 1) / 2
        pos += 1.5

    if not data:
        ax.text(0.5, 0.5, "no data yet", ha="center", va="center", transform=ax.transAxes)
        return fig, ax

    # Smooth violin body (paper-style fill) plus mean/min-max error bars and a median line,
    # all in black so they read clearly against the colored bodies.
    parts = ax.violinplot(data, positions=positions, showmeans=True, showextrema=True, showmedians=True, widths=0.8)
    for body, color in zip(parts["bodies"], colors):
        body.set_facecolor(color)
        body.set_edgecolor("black")
        body.set_linewidth(0.8)
        body.set_alpha(0.8)
    for key in ("cmins", "cmaxes", "cbars"):
        if key in parts:
            parts[key].set_color("black")
            parts[key].set_linewidth(1.0)
    if "cmeans" in parts:
        parts["cmeans"].set_color("black")
        parts["cmeans"].set_linestyle("--")
        parts["cmeans"].set_linewidth(1.2)
    if "cmedians" in parts:
        parts["cmedians"].set_color("black")
        parts["cmedians"].set_linewidth(1.5)

    ax.set_xticks(positions)
    ax.set_xticklabels(tick_labels)
    ax.set_xlabel(r"$\phi$")
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    y_top = ax.get_ylim()[1]
    for a, cx in group_centers.items():
        ax.text(cx, y_top, f"$\\alpha$={a:g}", ha="center", va="bottom", fontsize=9)

    handles = [plt.Line2D([0], [0], color=alpha_colors[a], lw=6, label=f"$\\alpha$={a:g}") for a in alphas]
    handles.append(plt.Line2D([0], [0], color="black", lw=1.2, linestyle="--", label="mean"))
    handles.append(plt.Line2D([0], [0], color="black", lw=1.5, label="median"))
    ax.legend(handles=handles, loc="upper left")

    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150)
    return fig, ax

In [ ]:
# Figure 7A reproduction: domain radius distributions (see the CLAHE-masking fix noted in the
# title cell -- radius is now a validated measurement, not an upper bound)
plot_grouped_violin(
    domain_radii_df, "radius_nm", r"$R_{d,i}$ (nm)",
    "SR-EV Nanodomain Radius\n",
    save_path="sr_ev_domain_radius_violins.png",
)
plt.show()

In [ ]:
# Domain COUNT distribution across replicate configs per (alpha, phi)
plot_grouped_violin(
    domain_counts_df, "n_domains", "Domains per 100 nm slab",
    "SR-EV Nanodomain Counts\n",
    save_path="sr_ev_domain_count_violins.png",
)
plt.show()